In [ ]:
from google import genai
from google.genai import types
import glob
import pandas as pd
from google.genai import types
from enum import Enum
from pydantic import BaseModel, Field
from datasets import load_dataset
import numpy as np
import json

client = genai.Client()

In [ ]:
MODEL_ID = "gemini-3.1-pro-preview"

In [ ]:
dataset = load_dataset("allenai/IFBench_test")["train"]
queries = dataset["prompt"]

In [ ]:
response_files = glob.glob("generations_ifbench/*/*")

In [ ]:
response_record = []
for r in response_files:
    responses = json.load(open(r, "r"))
    splits = r.split("/")
    model = splits[1]
    persona = splits[2].replace("_", " ")[:-5]
    response_record.extend([{"model": model, "persona": persona, "response": y, "prompt": p, "question_id": idx} for idx, (y, p) in enumerate(zip(responses, queries))])
test_df = pd.DataFrame(response_record)

In [ ]:
def split_base_and_variant(model_name: str) -> str:
    return model_name.replace("-SFT+DPO-v2", "")


def build_training_gap_table(df: pd.DataFrame) -> pd.DataFrame:
    # 1. Identify training status and normalize base model name
    df["is_trained"] = df["model"].str.contains(r"SFT\+DPO", regex=True)
    df["base_family"] = df["model"].apply(split_base_and_variant)

    # 2. Map boolean flags to descriptive column labels
    df["status"] = df["is_trained"].map(
        {False: "response_before", True: "response_after"}
    )

    # 3. Pivot response values into side-by-side columns
    pivoted = df.pivot(
        index=["base_family", "persona", "question_id", "prompt"],
        columns="status",
        values="response",
    ).reset_index()

    # Remove the column index name added by pivot
    pivoted.columns.name = None

    # Reorder columns cleanly
    columns_order = [
        "base_family",
        "persona",
        "question_id",
        "prompt",
        "response_before",
        "response_after",
    ]

    return pivoted[[col for col in columns_order if col in pivoted.columns]]

In [ ]:
test_df = build_training_gap_table(test_df)

In [ ]:
def swap_alternating_responses(df: pd.DataFrame) -> pd.DataFrame:
    # 1. Ensure clean, continuous 0..N-1 index for modulo checking
    df = df.reset_index(drop=True)

    # 2. Identify odd rows
    is_odd = df.index % 2 == 1

    # 3. Assign Model A and Model B based on row parity
    df["Model A"] = np.where(
        is_odd, df["response_after"], df["response_before"]
    )
    df["Model B"] = np.where(
        is_odd, df["response_before"], df["response_after"]
    )

    # 4. (Recommended) Keep ground truth column to unmask results later
    df["model_a_origin"] = np.where(is_odd, "trained", "base")

    # 5. Clean up old response columns
    df = df.drop(columns=["response_before", "response_after"])

    return df

In [ ]:
test_df = swap_alternating_responses(test_df)

In [ ]:
test_df

In [ ]:
len(queries) * 3 * 21

In [ ]:
def get_prompt(prompt, response_a, response_b):
    judge_instruction = f"""You are an expert, impartial AI judge evaluating two AI responses to the same user request.
Your primary job is to judge how well each response fulfills the SUBSTANTIVE, CREATIVE, or INFORMATIONAL core task of the prompt.

--- EVALUATION RULES ---
SUBSTANCE OVER MECHANICAL CONSTRAINTS:
- Primary focus: Task completion, reasoning depth, writing quality, accuracy, and adherence to core user intent.
- Secondary focus: Mechanical constraints (e.g., word count, specific keyword usage, formatting rules).
- NEVER reward a response that spams keywords or numbers to satisfy a constraint while failing to complete the main request.
--- INPUT DATA ---
User Request:
{prompt}

Response A:
{response_a}

Response B:
{response_b}"""
    return judge_instruction

In [ ]:
test_df["judge_prompt"] = test_df.apply(lambda x: get_prompt(x["prompt"], x["Model A"], x["Model B"]), axis=1)

In [ ]:
print(test_df.judge_prompt.tolist()[0])

In [ ]:
test_df

In [ ]:
test_df.reset_index(names="key")[["key", "base_family", "persona", "prompt", "Model A", "Model B", "model_a_origin"]].to_csv(f"index_mapping_ifbench.csv", index=False)

In [ ]:
class EvaluationScore(BaseModel):
    winner: str = Field(
        description="The winning response: strictly 'A' or 'B'"
    )

In [ ]:
request_data = [{"key": idx, "request": {"generation_config": {
                    "temperature": 0.0, 
                    "response_mime_type": "application/json",
                    'response_schema': EvaluationScore.model_json_schema()},
                    "contents": [{"parts": [{"text": p}]}]}} for idx, p in enumerate(test_df.judge_prompt.tolist())]

In [ ]:
request_data[-1]

In [ ]:
import json

json_file_path = 'batch_requests.json'

with open(json_file_path, 'w') as f:
    for req in request_data:
        f.write(json.dumps(req) + '\n')

# 2. Upload JSONL file to File API.
print(f"Uploading file: {json_file_path}")
uploaded_batch_requests = client.files.upload(
    file=json_file_path,
    config=types.UploadFileConfig(display_name='batch-input-file')
)
print(f"Uploaded file: {uploaded_batch_requests.name}")

In [ ]:
batch_job_from_file = client.batches.create(
    model=MODEL_ID,
    src=uploaded_batch_requests.name,
    config={
        'display_name': 'my-batch-job-from-file',
    }
)
print(f"Created batch job from file: {batch_job_from_file.name}")        

In [ ]:
import time

job_name = batch_job_from_file.name

print(f"Polling status for job: {job_name}")

# Poll the job status until it's completed.
while True:
    batch_job = client.batches.get(name=job_name)
    if batch_job.state.name in ('JOB_STATE_SUCCEEDED', 'JOB_STATE_FAILED', 'JOB_STATE_CANCELLED'):
        break
    print(f"Job not finished. Current state: {batch_job.state.name}. Waiting 30 seconds...")
    time.sleep(30)

print(f"Job finished with state: {batch_job.state.name}")
if batch_job.state.name == 'JOB_STATE_FAILED':
    print(f"Error: {batch_job.error}")

In [ ]:
if batch_job.state.name == 'JOB_STATE_SUCCEEDED':
    # The output is in another file.
    result_file_name = batch_job.dest.file_name
    print(f"Results are in file: {result_file_name}")

    print("\nDownloading and parsing result file content...")
    file_content_bytes = client.files.download(file=result_file_name)
    file_content = file_content_bytes.decode('utf-8')

    # Define the output directory and file path
    # (Make sure MODEL_ID is defined earlier in your script)
    output_dir = "./ratings"
    output_file_path = f"{output_dir}/{MODEL_ID}-ifbench_ratings.jsonl"
    
    # Open the file to write the JSONL content
    with open(output_file_path, 'w', encoding='utf-8') as f:
        # The result file is also a JSONL file. Parse and print each line.
        for line in file_content.splitlines():
            if line:
                parsed_response = json.loads(line)
                
                # Write the exact line (or re-serialized JSON) to the output file
                f.write(json.dumps(parsed_response) + '\n')
                
                # Pretty-print the JSON for readability
                print(json.dumps(parsed_response, indent=2))
                print("-" * 20)
                
    print(f"\nSuccessfully saved all responses to: {output_file_path}")
else:
    print(f"Job did not succeed. Final state: {batch_job.state.name}")